# B7 Monitor Implementation Only
All RUN flags default False. Formal run requires a separate Chat approval after review. Validation is COMPLETE_FROZEN. Monitor is NOT_RUN; OBSERVED_ONLY with no Validation override. This notebook is dormant until a separate explicit Colab approval. Freeze SHA is supplied explicitly after publication.

In [ ]:
from pathlib import Path
import os, sys, subprocess
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
REVIEWED_SHA = ''  # Paste Chat-reviewed Monitor Implementation Freeze SHA, never branch HEAD.
REVIEWED_PRODUCER_SHA = ''  # Finalize: explicitly pin the same reviewed Producer Freeze.
REPO = Path('/content/b7_monitor_repo')
DATA_ROOT = Path('/content/drive/MyDrive/REPLACE_WITH_FROZEN_DATA_ROOT')
LOCAL = Path('/content/b7_monitor_run')
DRIVE = Path('/content/drive/MyDrive/REPLACE_WITH_NEW_Monitor_CHECKPOINT_ROOT')
FINAL_OUTPUT = Path('/content/b7_monitor_finalize')  # Must not exist.
ARCHIVE = Path('/content/drive/MyDrive/REPLACE_WITH_NEW_Monitor_ARCHIVE_ROOT')
APPROVAL = ''  # Separate explicit Chat authorization is required.
RUN_MOUNT = False
RUN_INSTALL = False
RUN_CHECKOUT = False
RUN_INPUT_AUDIT = False
RUN_M1_AUDIT = False
RUN_TESTS = False
RUN_SMOKE = False
RUN_PREFLIGHT = False
RUN_FORMAL = False
RUN_RESUME = False
RUN_FINALIZE_ONLY = False
RUN_ARCHIVE = False

if RUN_FINALIZE_ONLY:
    assert not any([RUN_FORMAL,RUN_RESUME,RUN_M1_AUDIT,RUN_TESTS,RUN_SMOKE,RUN_PREFLIGHT])
RUN_CALENDAR_AUDIT = False


## Mount

In [ ]:
if RUN_MOUNT:
    from google.colab import drive
    drive.mount('/content/drive')


## Install pinned dependencies

In [ ]:
if RUN_INSTALL:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'numpy==2.3.5', 'pandas==2.2.3'], check=True)
    print('Restart runtime if these packages were already imported; then rerun configuration.')


## Checkout reviewed commit

In [ ]:
if RUN_CHECKOUT:
    import re
    assert re.fullmatch('[0-9a-f]{40}', REVIEWED_SHA)
    if not REPO.exists():
        subprocess.run(['git','clone','https://github.com/TR-KJ/time-entry-portfolio-lab.git',str(REPO)],check=True)
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip()
    subprocess.run(['git','-C',str(REPO),'fetch','origin',REVIEWED_SHA],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','--detach',REVIEWED_SHA],check=True)
    assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==REVIEWED_SHA
    sys.path.insert(0,str(REPO/'src/research'))


## Input audit

In [ ]:
if RUN_INPUT_AUDIT:
    from b7.monitor_input import input_config
    _, data = input_config()
    print({'Status':'PASS','CandidateCount':len(data['Candidates'])})


## Frozen event calendar identity audit


In [ ]:
if RUN_CALENDAR_AUDIT:
    from b7.u10_calendar import audit_calendar
    c = audit_calendar()
    print({'Status':'PASS','Events':len(c['Events']),'CalendarSourceCommit':c['CalendarSourceCommit']})


## M1 audit

In [ ]:
if RUN_M1_AUDIT:
    from b7.stage1_input import resolve_exact, audit_inputs
    paths = resolve_exact(DATA_ROOT)
    print({'Status':'PASS','ExactFiles':len(audit_inputs(paths))})


## Synthetic tests

In [ ]:
if RUN_TESTS:
    from b7.stage1_runtime import run_tests
    print(run_tests())


## Synthetic 2026 smoke only

In [ ]:
if RUN_SMOKE:
    from b7.monitor_smoke import run_smoke
    print(run_smoke())


## Preflight

In [ ]:
if RUN_PREFLIGHT:
    from b7.stage1_input import resolve_exact
    from b7.monitor_runtime import preflight
    proof = preflight(resolve_exact(DATA_ROOT), REVIEWED_SHA)
    print({'Status':proof['Status']})


## Formal 22 jobs

In [ ]:
if RUN_FORMAL:
    assert not RUN_RESUME and not RUN_FINALIZE_ONLY
    from b7.stage1_input import resolve_exact
    from b7.monitor_runtime import run_formal
    review = run_formal(resolve_exact(DATA_ROOT),REVIEWED_SHA,LOCAL,DRIVE,approval=APPROVAL)
    print({"CompletedJobs": review["CompletedJobs"], "ExpectedJobs": 22})  # No individual results before archive.


## Exact resume

In [ ]:
if RUN_RESUME:
    assert not RUN_FORMAL and not RUN_FINALIZE_ONLY
    from b7.stage1_input import resolve_exact
    from b7.monitor_runtime import run_formal
    review = run_formal(resolve_exact(DATA_ROOT),REVIEWED_SHA,LOCAL,DRIVE,approval=APPROVAL,resume=True)
    print({"CompletedJobs": review["CompletedJobs"], "ExpectedJobs": 22})  # No individual results before archive.


## Finalize-Only

In [ ]:
if RUN_FINALIZE_ONLY:
    assert not any([RUN_FORMAL,RUN_RESUME,RUN_M1_AUDIT,RUN_TESTS,RUN_SMOKE,RUN_PREFLIGHT])
    from b7.monitor_finalize_only import finalize_from_completed_jobs
    review=finalize_from_completed_jobs(DRIVE,FINAL_OUTPUT,REVIEWED_SHA,REVIEWED_PRODUCER_SHA,approval=APPROVAL,progress=print)
    print({"CompletedJobs": review["CompletedJobs"], "ExpectedJobs": 22})  # No individual results before archive.


## Archive after COMPLETE only
Create and verify the fresh Drive archive before opening candidate_results or displaying individual Monitor observations. Retain the original checkpoint root unchanged.

In [ ]:
if RUN_ARCHIVE:
    from b7.monitor_runtime import archive_completed
    archive_completed(FINAL_OUTPUT if RUN_FINALIZE_ONLY else LOCAL/'final',ARCHIVE)
    print({"ArchiveVerified": True, "IndividualResultsMayBeReviewed": True})
